In [ ]:
%pip install tqdm
%pip install joblib
%pip install tensorflow
%pip install -U hats lsdb
%pip install "dask[complete]"
%pip install 'light-curve[full]'

In [24]:
import os
import time
import warnings
import numpy as np
import pandas as pd
from tqdm import tqdm
import tensorflow as tf
from lsdb import read_hats
from dask.distributed import Client
from joblib import Parallel, delayed
from nested_pandas import NestedDtype
from joblib import wrap_non_picklable_objects

warnings.filterwarnings(action="ignore") 


In [37]:
def _float_feature(list_of_floats):  # float32
    return tf.train.Feature(float_list=tf.train.FloatList(value=list_of_floats))


def get_record(id_, label, last_index, numpy_lc):
    """
    Create tf.records from numpy values.

    Parameters:
        id_ (int): object id
        label (str): class 
        numpy_lc (numpy array): time, magnitudes, mag error, band info

    Returns:
        tensorflow record
    """
    dict_sequence = {}
    dict_features={
    'id': tf.train.Feature(int64_list=tf.train.Int64List(value=[id_])),
    'label': tf.train.Feature(bytes_list=tf.train.BytesList(value=[str(label).encode()])),
    'last_index': tf.train.Feature(int64_list=tf.train.Int64List(value=list(last_index.values()))),
    'bands': tf.train.Feature(bytes_list=tf.train.BytesList(value=[k.encode('utf-8') for k in last_index.keys()])) 
    }
    
    element_context = tf.train.Features(feature = dict_features)

    for col in range(numpy_lc.shape[1]):
        seqfeat = _float_feature(numpy_lc[:, col])
        seqfeat = tf.train.FeatureList(feature = [seqfeat])
        dict_sequence['dim_{}'.format(col)] = seqfeat

    element_lists = tf.train.FeatureLists(feature_list=dict_sequence)
    rec = tf.train.SequenceExample(context = element_context,
                                  feature_lists= element_lists)
    return rec


def train_test_split(frame, train_size):
    """
    Divide the dataset into train, validation and test subsets.
    Note:
        val_size = 1 - train_size
        test_size = train_size + val_size (entire dataset)

    Paremeters:
        frame (Dataframe): Dataframe following the astro-standard format
        train_size (float): train fraction

    Returns:
        tuple : (name of subset, subframe with metadata)
    """

    frame = frame.sample(frac=1)
    n_samples = frame.shape[0]

    n_train = int(n_samples*train_size)
    sub_test  = frame.iloc[:]
    sub_train = frame.iloc[:n_train]
    sub_val   = frame.iloc[n_train:]
    
    return ('train', sub_train), ('val', sub_val), ('test', sub_test)



def create_record(id_, label, last_index, numpy_lc, writer):
    """
    Parameters:
    id_ (int): index of the lightcurve
    label (str): class name
    last_index (dict): last index of each band
    numpy_lc (numpy array): lightcurve data
    """
    try:
        rec = get_record(id_, label, last_index, numpy_lc)
        writer.write(rec.SerializeToString())
    except Exception as e:
        print('[INFO] {} could not be processed. Create Rec. Exception Raised - {}'.format(id_, e))


@wrap_non_picklable_objects
def process_lc(id_, row):
    """
    Process each light curve

    Parameters:
    id_ (int): index of the lightcurve
    row (DataFrame): lightcurve data
    ------------------------------------------------
    Return:
    id_ (int): index of the lightcurve
    label (str): class name
    last_index (dict): last index of each band
    numpy_lc (numpy array): lightcurve data
    """
    
    try:
        #
        # Store the last index of each band 
        #
        last_index = dict()
        #
        # Define the ZTF bands in Angstroms (ZTF mean filter wavelengths)
        #
        bands = {'g': np.log10(4835.99), 'r': np.log10(6468.25), 'i': np.log10(7993.65)}
        #
        # Store the label of the lightcurve
        #
        label = row["Class"]
        #
        # Filter the "lc" column from the catalog and convert it to a DataFrame
        #
        row_ = row["lc"]  
        temp_df = pd.DataFrame(row_)
        #
        # Filter the lightcurve with no calibration errors and good quality observations 
        #
        mask_id = (temp_df["info"] == 0) & (temp_df["flag"] == 0)
        temp_df = temp_df[mask_id]
        if not temp_df.isnull().values.all():  
            #
            # Sort the lightcurve by band in the following order: g, r, i
            #
            temp_df['band_sorted'] = pd.Categorical( temp_df['band'], 
                                            categories=list(bands.keys()), 
                                            ordered=True
                                            )
            temp_df = temp_df.sort_values('band_sorted')
            temp_df = temp_df.reset_index(drop=True)
            temp_df.drop(columns=['band'], inplace=True)
            #
            # Store the last index of each band in the dictionary "last_index"
            #
            for k, v in bands.items():
                    
                    n_samples = temp_df[temp_df["band_sorted"] == k].shape[0]
                    
                    if k == 'g':
                        end_idx = n_samples-1
                        last_index[k] = end_idx
                    else:
                        end_idx = n_samples + end_idx
                        last_index[k] = end_idx
                        
            temp_df = temp_df.replace({"band_sorted":bands})
            temp_df = temp_df[["mjd", "mag", "magerr", "band_sorted"]]
            numpy_lc = temp_df.values
            #
            return id_, label, last_index, numpy_lc

        
    except Exception as e:
        print('[INFO] {} could not be processed. processed lc. Exception Raised - {}'.format(id_, e))
        return None
        
        
def write_records(frame, dest, max_lcs_per_chunk, n_jobs):
    """
    Get frames with fixed number of lightcurves in each chunk

    Parameters:
    -----------------------------------------------
    frame (dataframe): catalog data
    dest (str): path to store the files
    n_jobs (int): # of parallel jobs
    max_lcs_per_chunk (int): # of lcs to be stored in a chunk

    """
    #
    # 
    #
    collection = [frame.iloc[i:i+max_lcs_per_chunk] \
                  for i in range(0, frame.shape[0], max_lcs_per_chunk)]

    for counter, subframe in enumerate(collection):
        var = Parallel(n_jobs=n_jobs)(delayed(process_lc)(id_, row) \
                                    for id_, row in subframe.iterrows())

        with tf.io.TFRecordWriter(dest+'/chunk_{}.record'.format(counter)) as writer:
            for _, data_lc in enumerate(var):
                if data_lc is not None:
                    create_record(*data_lc, writer)


In [38]:
def create_dataset(df,
                    target,
                    njobs,
                    train_size=0.80,
                    max_lcs_per_chunk=100):
    
    """
    Create tf.records from the catalog

    Parameters: 
    ---------------------------------------------------------
    df (DataFrame): contains the catalog file
    target (str): directory path for the files to be stored
    n_jobs (int): # of parallel jobs
    train_size (float): train fraction
    max_lcs_per_chunk (int): # of lcs to be stored in a chunk
    """
    #
    #
    #
    info_df = pd.DataFrame()
    LC_COLUMN = "lc"
    #
    #
    #
    try: 
        if not train_size or train_size > 1:
            raise AssertionError(f"Please provide a valid train_size fraction. Got {train_size}")
        
        if not os.path.exists(target):
            os.makedirs(target, exist_ok=True)
    
        df = df.assign(**{LC_COLUMN: df[LC_COLUMN].astype(NestedDtype.from_pandas_arrow_dtype(df.dtypes[LC_COLUMN]))},)
        df = df.dropna(subset=['lc'])
        #
        # Save the number of classes and their counts in a .CSV file
        #
        unique, counts = np.unique(df['Class'], return_counts=True)
        info_df['label'] = unique
        info_df['size'] = counts
        info_df.to_csv(os.path.join(target, 'objects.csv'), index=False)
        #
        # Separate by class
        #
        cls_groups = df.groupby('Class')
        #
        # Write lcs in the records
        #
        for cls_name, cls_meta in tqdm(cls_groups, total=len(cls_groups)):
            subsets = train_test_split(cls_meta, train_size=train_size)

            for subset_name, frame in subsets:
                if frame is None:
                    continue
                dest = os.path.join(target, subset_name, cls_name)
                os.makedirs(dest, exist_ok=True)
                write_records(frame, dest, max_lcs_per_chunk, n_jobs=njobs)

    except Exception as e:
        print(f"\nException Raised: {e}\n")
        


In [ ]:
#
# Read catalog
#
read_catalog = read_hats('./cephids/zubercal_vcep', )
#    
print("\nStarting!")
start = time.time()
#
#
#
path_to_store="./cephids/zubercal_vcep"
#
#
#
catalog_compute = read_catalog._ddf.map_partitions(create_dataset, 
                                                        target=path_to_store,
                                                        njobs=4,
                                                        train_size=0.80,
                                                        max_lcs_per_chunk=100)

with Client() as client:
    display(client)
    catalog_compute.compute()

end = time.time()
print("\nTime (mins):", ((end-start)//60))
print("\nDone!")

    